# Notebook 2 — LSTM Traffic Prediction
Architecture: LSTM(64) → Dropout(0.2) → LSTM(32) → Dense(16) → Dense(1)

Callbacks: EarlyStopping · ReduceLROnPlateau · ModelCheckpoint

Metrics: RMSE · MAE · MAPE · R²

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

try:
    import numpy as np
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy"])
    import importlib
    np = importlib.import_module('numpy')
try:
    import plotly.graph_objects as go
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly"])
    go = importlib.import_module('plotly.graph_objects')
from plotly.subplots import make_subplots

from modules.predictor import build_lstm, train_lstm, evaluate_lstm, load_lstm
from config import LSTM_MODEL_PATH
import subprocess, importlib
import subprocess, sys, importlib

print('TensorFlow ready.')

NameError: name 'importlib' is not defined

## 1. Load Preprocessed Splits

In [ ]:
X_tr = np.load('../models/X_tr.npy')
X_te = np.load('../models/X_te.npy')
y_tr = np.load('../models/y_tr.npy')
y_te = np.load('../models/y_te.npy')

# Use last 10% of training as validation
val_split = int(len(X_tr) * 0.9)
X_val, y_val = X_tr[val_split:], y_tr[val_split:]
X_tr2, y_tr2 = X_tr[:val_split], y_tr[:val_split]

print(f'Train: {X_tr2.shape}  Val: {X_val.shape}  Test: {X_te.shape}')

## 2. Model Summary

In [ ]:
model = build_lstm()
model.summary()

## 3. Train

In [ ]:
model, history = train_lstm(X_tr2, y_tr2, X_val, y_val)
print(f'Best val_loss: {min(history.history["val_loss"]):.6f}')

## 4. Training Curves

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss (MSE)', 'MAE'))
epochs = list(range(1, len(history.history['loss']) + 1))

for col, metric in enumerate(['loss', 'mae'], start=1):
    fig.add_trace(go.Scatter(x=epochs, y=history.history[metric],
                             name=f'Train {metric}', line=dict(color='#00B4D8')), row=1, col=col)
    fig.add_trace(go.Scatter(x=epochs, y=history.history[f'val_{metric}'],
                             name=f'Val {metric}', line=dict(color='#F77F00', dash='dash')), row=1, col=col)

fig.update_layout(template='plotly_dark', height=350, title='LSTM Training Curves')
fig.show()

## 5. Evaluation Metrics

In [ ]:
metrics = evaluate_lstm(model, X_te, y_te)
print('=== Test Set Metrics ===')
for k, v in metrics.items():
    if k != 'y_pred':
        print(f'  {k:6s}: {v:.4f}')

## 6. Prediction vs Actual

In [ ]:
y_pred = metrics['y_pred']
idx    = list(range(len(y_te)))

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=idx, y=y_te,   name='Actual',    line=dict(color='#00B4D8')))
fig2.add_trace(go.Scatter(x=idx, y=y_pred, name='Predicted', line=dict(color='#F77F00', dash='dash')))
fig2.update_layout(title='LSTM: Actual vs Predicted Traffic',
                   xaxis_title='Sample', yaxis_title='Normalised Traffic',
                   template='plotly_dark', height=350)
fig2.show()

print(f'Model saved to: {LSTM_MODEL_PATH}')